In [0]:
%sql

select * from medallion_project.bronze01.trips_raw_with_al;

In [0]:
from pyspark.sql.functions import col, lit, to_timestamp, cast

trips1 = spark.read.table('medallion_project.bronze01.trips_raw')
trips2 = spark.read.table('medallion_project.bronze01.trips_raw_with_al')

display(trips1)

#Casting trips2 (trips_raw_with_al - from auto-loader). It was loaded as plain text

In [0]:
from pyspark.sql.functions import col, expr
from functools import reduce
from operator import or_

# dictionary of casting, if it is string then skipped in mapping
cast_mapping = {
    "VendorID": "integer",
    "tpep_pickup_datetime": "timestamp",
    "tpep_dropoff_datetime": "timestamp",
    "passenger_count": "integer",
    "trip_distance": "double",
    "RatecodeID": "integer",
    "PULocationID": "integer",
    "DOLocationID": "integer",
    "payment_type": "integer",
    "fare_amount": "double",
    "extra": "double",
    "mta_tax": "double",
    "tip_amount": "double",
    "tolls_amount": "double",
    "improvement_surcharge": "double",
    "total_amount": "double",
    "congestion_surcharge": "double"
}

# 1. generate list of conditions per column
error_conditions = [
    (col(c).isNotNull() & expr(f"try_cast({c} AS {target_type})").isNull())
    for c, target_type in cast_mapping.items()
]

# 2. merge all conditions with OR operator
combined_error_condition = reduce(or_, error_conditions)

# 3. filter wrong rows
df_all_errors = trips2.filter(combined_error_condition)

# display wrong rows if exist
display(df_all_errors)    

In [0]:
# if no rows then cast all cols
if df_all_errors.count() == 0:
    columns_to_select = [
        col(c).cast(cast_mapping[c]).alias(c) if c in cast_mapping else col(c) 
        for c in trips2.columns
    ]
    df_trips2_casted = trips2.select(*columns_to_select)
else:
    df_trips2_casted = trips2

display(df_trips2_casted)


# Union both tables in one

In [0]:
trips = trips1.unionByName(df_trips2_casted, allowMissingColumns=True)
# display(trips.filter(col("tpep_pickup_datetime") > lit("2019-07-01")))

In [0]:
display(trips.summary())

# Drop duplicates by surrogate key combination of: 
    "tpep_pickup_datetime",
    "tpep_dropoff_datetime",
    "trip_distance",
    "PULocationID",
    "DOLocationID"

In [0]:
from pyspark.sql.functions import col

dedup_cols = [
    "tpep_pickup_datetime",
    "tpep_dropoff_datetime",
    "trip_distance",
    "PULocationID",
    "DOLocationID"
    ]

count_before = trips.count()

trips_duplicates_count = (trips
    .groupBy(dedup_cols)
    .count()
    .filter(col("count") > 1)
    .orderBy(col("count").desc())
)

trips_deduped = trips.dropDuplicates(subset=dedup_cols)
count_after = trips_deduped.count()
# display(trips_duplicates_count)
# print(count_before, trips_duplicates_count)
print(f"Amount of deleted duplicates is: {count_before - count_after}")

# Remove records with null value in key column


In [0]:
from pyspark.sql import functions as F

count_na = (trips_deduped
            .filter(F.col("tpep_pickup_datetime").isNull() |
                    F.col("tpep_dropoff_datetime").isNull())
            .count()
            )

trips_silver_cleansed = trips_deduped.dropna(subset=["tpep_pickup_datetime", "tpep_dropoff_datetime"])
print(count_na)

#Fill null and nonsense values

In [0]:
# display(trips_cleansed_null)
df = trips_silver_cleansed.filter(col("passenger_count") == 0)
display(df)
print(df.count())
print(trips_silver_cleansed.filter(col("passenger_count").isNull()).count())



In [0]:
from pyspark.sql.functions import col, sum, when

# Generuje jeden wiersz z liczbą wartości NULL dla każdej kolumny
df_null_counts = trips.select([
    sum(when(col(c).isNull(), 1).otherwise(0)).alias(c)
    for c in trips.columns
])

# Sprawdzenie rozkładu dla kluczowych kolumn liczbowych
display(trips.select("passenger_count", "fare_amount", "trip_distance", "total_amount").summary())

In [0]:
# Passenger count fill 0 with Null value because its impossible to perform ride without passenger (I omit situation to have 0 passenger cause of ie packages travel etc)

# Zamiana wartości 0 na NULL, pozostawienie obecnych NULLi oraz poprawnych wartości bez zmian
df_silver = trips_silver_cleansed.withColumn(
    "passenger_count",
    when(col("passenger_count") == 0, lit(None).cast("integer"))
    .otherwise(col("passenger_count"))
)

df_silver = df_silver.filter((col("fare_amount") < 0) & ~(col("payment_type").isin([4, 6])))
display(df_silver)

Silver prod ready

In [0]:
from pyspark.sql.functions import col, when, lit

df = (trips_deduped
      # 1. remove wrong rows
      .filter(
          col("tpep_pickup_datetime").isNull() |
          col("tpep_dropoff_datetime").isNull() |
          col("tpep_pickup_datetime") > col("tpep_dropoff_datetime") |
          col("fare_amount")
      )
      
      
      )